## Geo Alignment Debug Log

Early attempts at matching locations to delegation codes.

In [ ]:
# Load raw data
import pandas as pd
from pathlib import Path

raw = Path("../data/raw")
df = pd.read_csv(raw / "tunisia-real-estate.csv")
print(f"Loaded {len(df)} rows")

In [ ]:
# List some locality values
print("Sample localities:")
print(df['Locality'].value_counts().head(20))

First issue: "Ariana" vs "Ariana Ville" vs "Ariana Ville-"

In [ ]:
# Load delegation reference
GEO_DIR = Path("../geo")
delegations = pd.read_csv(GEO_DIR / "delegations.csv")
print(f"Reference delegations: {len(delegations)}")
print(delegations['name_fr'].head(10).tolist())

## Attempt 1: Simple string matching

In [ ]:
# Try exact match
def match_exact(loc, ref_list):
    loc_clean = str(loc).strip().lower()
    ref_clean = [str(r).strip().lower() for r in ref_list]
    if loc_clean in ref_clean:
        return ref_list[ref_clean.index(loc_clean)]
    return None

df['matched_deleg'] = df['Locality'].apply(lambda x: match_exact(x, delegations['name_fr']))
matched = df['matched_deleg'].notna().sum()
print(f"Exact matches: {matched} / {len(df)} = {matched/len(df)*100:.1f}%")

# Terrible! Only 12%

## Attempt 2: Fuzzy with SequenceMatcher

In [ ]:
# Try fuzzy matching
from difflib import SequenceMatcher

def fuzzy_match(loc, ref_list, threshold=0.6):
    if pd.isna(loc): return None
    loc_str = str(loc).strip().lower()
    best_ratio = 0
    best_match = None
    for ref in ref_list:
        ref_str = str(ref).strip().lower()
        ratio = SequenceMatcher(None, loc_str, ref_str).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best_match = ref
    return best_match if best_ratio >= threshold else None

df['matched_deleg'] = df['Locality'].apply(lambda x: fuzzy_match(x, delegations['name_fr']))
matched = df['matched_deleg'].notna().sum()
print(f"Fuzzy matches: {matched} / {len(df)} = {matched/len(df)*100:.1f}%")

# Better: ~52% but still many misses

## Attempt 3: Add governorate hints

In [ ]:
# Map governorates first, then match within governorate
governorates = pd.read_csv(GEO_DIR / "governorates.csv")
gov_map = dict(zip(governorates['name_fr'], governorates['delegations']))

# Not implemented here - next iteration

## Attempt 4: Add alias mappings

In [ ]:
# Common aliases from inspecting failures
ALIASES = {
    "ariana ville": "Ariana",
    "la soukra": "La Soukra",
    "le bardo": "Le Bardo",
    "sahloul": "Sahloul",
    "tunis": "Tunis",
    "tunis ville": "Tunis",
    "sfax": "Sfax",
    "sfax ville": "Sfax"
}

# Apply aliases before matching
def apply_alias(loc):
    if pd.isna(loc): return None
    loc_lower = str(loc).strip().lower()
    return ALIASES.get(loc_lower, loc_lower)

df['Locality_clean'] = df['Locality'].apply(apply_alias)
# Then apply fuzzy match...

## Summary

This was the geo debugging notebook from early iterations.